# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-morad15/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
from huggingface_hub import login
import duckdb

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

print("HF_TOKEN loaded successfully.")

# Authenticate with Hugging Face
login(token=HF_TOKEN, add_to_git_credential=False)

# Start DuckDB
con = duckdb.connect()

print("DuckDB ready.")
print("Hugging Face authentication ready.")

HF_TOKEN loaded successfully.
DuckDB ready.
Hugging Face authentication ready.


In [3]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("Warehouse relation ready.")

Warehouse relation ready.


In [4]:
REL_FEB = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
"""

REL_MAR = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("February and March warehouse relations ready.")

February and March warehouse relations ready.


In [5]:
con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("DuckDB Hugging Face secret configured.")

DuckDB Hugging Face secret configured.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The label comes from the portfolio trend direction, calculated from the 30-day impression change versus the previous 30 days. Pages are labeled as growing or declining based on that observed change.

The comparison shows that growing pages were younger and longer on average than declining pages. The large sample makes the observed differences useful as directional evidence.

However, the validation design does not fully carry a causal claim. This is an observational comparison, so it shows that age and word count are associated with different trend groups, but it does not prove that making a page longer or younger will cause growth. A stronger validation would use a time-aware holdout or a before/after refresh comparison to test whether changes in these factors are followed by measurable performance changes.

### Finding 3 — Click Capture by Position Tier

The label comes from position tiers derived from average search position. CTR is calculated as weighted clicks divided by impressions for each tier.

The result shows a clear measured difference in click capture: weighted CTR is highest in the Top 3 and falls substantially for deeper positions. This is a useful descriptive portfolio comparison because it is based on aggregate clicks and impressions rather than row-level average CTR.

The validation design supports the descriptive claim, but it does not establish that moving a page to a higher position will automatically cause the observed CTR increase. Position and CTR are measured outcomes that can also be affected by query intent, SERP features, and other factors. The finding is therefore best used as decision-support for prioritizing visible pages, not as a causal rule.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest validation result

My Week-5 model already used a client-level holdout, so the original result was already grouped by client rather than split randomly by row.

The original Random Forest achieved Precision@50 = 0.82, while the baseline achieved 0.02.

For this audit, I re-run the same decision-time features under the same client-level separation and check that no client appears in both training and validation. This is an appropriate grouped split because the decision is intended to generalize across content pages while reducing the risk of learning client-specific patterns.

The comparison is therefore interpreted as a validation check rather than a claim of causal performance.


In [6]:
# SHonest grouped validation

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

TARGET = "review_priority"
RANDOM_STATE = 42


def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


# Reuse baseline_df from the earlier notebook cells if available.
# If not available, rebuild it from February and March relations.

if "baseline_df" not in globals():

    baseline_df = con.sql(f"""
    WITH feb AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_avg_position) AS gsc_avg_position,
            SUM(gsc_impressions) AS gsc_impressions,
            SUM(gsc_clicks) AS gsc_clicks,
            SUM(ga4_sessions) AS ga4_sessions,
            SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
        FROM {REL_FEB}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_avg_position) AS march_avg_position
        FROM {REL_MAR}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        f.*,
        m.march_avg_position,

        CASE
            WHEN m.march_avg_position > f.gsc_avg_position
            THEN 1
            ELSE 0
        END AS review_priority

    FROM feb f
    INNER JOIN march m
        ON f.client_hash_id = m.client_hash_id
        AND f.content_hash_id = m.content_hash_id

    WHERE f.gsc_avg_position IS NOT NULL
      AND m.march_avg_position IS NOT NULL
    """).df()


# Grouped split by client
clients = baseline_df["client_hash_id"].dropna().unique()

train_clients, val_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_df = baseline_df[
    baseline_df["client_hash_id"].isin(train_clients)
].copy()

val_df = baseline_df[
    baseline_df["client_hash_id"].isin(val_clients)
].copy()


# Leakage-safe feature/target separation
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[FEATURES].copy()
y_val = val_df[TARGET].copy()


# Train Random Forest
rf_honest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_honest.fit(X_train, y_train)

val_prob = rf_honest.predict_proba(X_val)[:, 1]

honest_precision_50 = precision_at_k(
    val_prob,
    y_val,
    k=50
)


# Verify client separation
client_overlap = set(train_df["client_hash_id"]).intersection(
    set(val_df["client_hash_id"])
)

print("Total clients:", len(clients))
print("Train clients:", len(train_clients))
print("Validation clients:", len(val_clients))
print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Client overlap:", len(client_overlap))
print("Honest Precision@50:", round(honest_precision_50, 4))

assert len(client_overlap) == 0
assert TARGET not in FEATURES

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total clients: 42
Train clients: 33
Validation clients: 9
Train rows: 121644
Validation rows: 12594
Client overlap: 0
Honest Precision@50: 0.8


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


The final feature set contains only February decision-time signals:

* GSC impressions
* GSC clicks
* GSC average position
* GA4 sessions
* GA4 engaged sessions

The target `review_priority` is created using the March average position, so March outcome information is kept out of the feature matrix.

I also check that the target is not included in the feature list, that the training and validation clients do not overlap, and that each client-content pair appears only once in the modeling table.

The audit therefore finds no direct target leakage in the final five-feature model. The remaining limitation is that this is an observational decision-support model, so the validation result should not be interpreted as causal evidence.


In [7]:

print("FEATURES:")
for feature in FEATURES:
    print(" -", feature)

print("\nTARGET:")
print(" -", TARGET)


# 1. Target must not be a feature
target_in_features = TARGET in FEATURES
print("\nTarget included in features:", target_in_features)

assert not target_in_features


# 2. March outcome columns must not be features
future_terms = ["march", "review_priority"]

future_features = [
    feature
    for feature in FEATURES
    if any(term in feature.lower() for term in future_terms)
]

print("Future/outcome-like features:", future_features)

assert len(future_features) == 0


# 3. Check client overlap
train_clients_check = set(train_df["client_hash_id"])
val_clients_check = set(val_df["client_hash_id"])

client_overlap = train_clients_check.intersection(
    val_clients_check
)

print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0


# 4. Check duplicate client-content pairs
pair_duplicates = baseline_df.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print("Duplicate client-content pairs:", pair_duplicates)

assert pair_duplicates == 0


# 5. Check missing values in final features
missing_features = X_train[FEATURES].isna().sum()

print("\nMissing values in training features:")
print(missing_features)


# 6. Final audit summary
print("\nLeakage audit passed.")

FEATURES:
 - gsc_impressions
 - gsc_clicks
 - gsc_avg_position
 - ga4_sessions
 - ga4_engaged_sessions

TARGET:
 - review_priority

Target included in features: False
Future/outcome-like features: []
Client overlap: 0
Duplicate client-content pairs: 0

Missing values in training features:
gsc_impressions             0
gsc_clicks                  0
gsc_avg_position            0
ga4_sessions            64582
ga4_engaged_sessions    64582
dtype: int64

Leakage audit passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


**Original claim:**

The Random Forest model is much better than the baseline and can identify the pages that should receive review priority.

**Safer claim:**

In the evaluated client-level holdout, the Random Forest achieved a measured Precision@50 of 0.82 compared with 0.02 for the baseline rule. This observed difference suggests that the model can provide useful directional decision-support for prioritizing content pages for review in this dataset. The result is not evidence that the model will generalize to every client or that the model causes better SEO outcomes.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.